In [1]:

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

categories = [
    'comp.graphics',
    'rec.sport.baseball'
]


newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers','footers','quotes')
)



print("Dataset Loaded Successfully")
print("Number of Documents:", len(newsgroups.data))


X_text = newsgroups.data
y = newsgroups.target



print("Classes:", newsgroups.target_names)


X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


vectorizer = CountVectorizer(
    stop_words='english',
    max_features=5000
)


X_train = vectorizer.fit_transform(
    X_train_text
)


X_test = vectorizer.transform(
    X_test_text
)



V = X_train.shape[1]


print("\nVocabulary Size:", V)


classes = np.unique(y_train)


mle_parameters = {}


class_prior = {}



for c in classes:


    X_class = X_train[y_train == c]


    word_counts = np.asarray(
        X_class.sum(axis=0)
    ).flatten()



    total_words = word_counts.sum()



    # MLE probability
    theta_mle = (
        word_counts /
        total_words
    )


    mle_parameters[c] = theta_mle



    # Class probability
    class_prior[c] = (
        X_class.shape[0] /
        X_train.shape[0]
    )



    print("\nClass:", newsgroups.target_names[c])

    print(
        "Total Words:",
        total_words
    )


    print(
        "Zero Probability Words (MLE):",
        np.sum(theta_mle == 0)
    )



print("\nMLE Estimation Completed")


alphas = [
    0.5,
    1,
    5,
    10
]


map_parameters = {}



for alpha in alphas:


    print("\n==============================")
    print("Dirichlet Prior alpha =", alpha)
    print("==============================")


    map_parameters[alpha] = {}



    for c in classes:


        X_class = X_train[y_train == c]


        word_counts = np.asarray(
            X_class.sum(axis=0)
        ).flatten()



        total_words = word_counts.sum()



        # MAP with Dirichlet prior
        theta_map = (
            word_counts + alpha
        ) / (
            total_words + alpha * V
        )



        map_parameters[alpha][c] = theta_map



        print(
            "Class:",
            newsgroups.target_names[c]
        )


        print(
            "Zero Probability Words:",
            np.sum(theta_map == 0)
        )



print("\nMAP Estimation Completed")


def predict_multinomial(
        X,
        parameters,
        priors
):


    predictions = []


    log_parameters = {}


    for c in classes:

        log_parameters[c] = np.log(
            parameters[c] + 1e-12
        )



    for doc in X:


        scores = []


        for c in classes:


            score = (
                np.log(priors[c])
                +
                np.sum(
                    doc.toarray()
                    *
                    log_parameters[c]
                )
            )


            scores.append(score)



        predictions.append(
            classes[
                np.argmax(scores)
            ]
        )


    return np.array(predictions)

mle_prediction = predict_multinomial(
    X_test,
    mle_parameters,
    class_prior
)



mle_accuracy = accuracy_score(
    y_test,
    mle_prediction
)



print("\n============================")
print("MLE Performance")
print("============================")


print(
    "Accuracy:",
    mle_accuracy
)



results = []


results.append(
    [
        "MLE",
        "-",
        mle_accuracy
    ]
)



for alpha in alphas:


    prediction = predict_multinomial(
        X_test,
        map_parameters[alpha],
        class_prior
    )


    accuracy = accuracy_score(
        y_test,
        prediction
    )



    results.append(
        [
            "MAP",
            alpha,
            accuracy
        ]
    )



    print(
        "\nMAP alpha =",
        alpha
    )


    print(
        "Accuracy:",
        accuracy
    )

comparison = pd.DataFrame(
    results,
    columns=[
        "Method",
        "Alpha",
        "Accuracy"
    ]
)



print("\n============================")
print("MLE vs MAP Comparison")
print("============================")


print(comparison)


best_alpha = max(
    alphas,
    key=lambda a:
    comparison[
        comparison["Alpha"] == a
    ]["Accuracy"].values[0]
)



best_prediction = predict_multinomial(
    X_test,
    map_parameters[best_alpha],
    class_prior
)



print(
    "\nBest MAP Alpha:",
    best_alpha
)



print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        best_prediction
    )
)


feature_names = vectorizer.get_feature_names_out()



def top_words(theta, n=10):

    indexes = np.argsort(theta)[-n:]

    return pd.DataFrame(
        {
            "Word":
            feature_names[indexes],

            "Probability":
            theta[indexes]
        }
    )



for c in classes:


    print(
        "\nClass:",
        newsgroups.target_names[c]
    )


    print("\nTop words using MLE")

    print(
        top_words(
            mle_parameters[c]
        )
    )


    print(
        "\nTop words using MAP alpha=10"
    )


    print(
        top_words(
            map_parameters[10][c]
        )
    )



print("\nProgram Completed Successfully")

Dataset Loaded Successfully
Number of Documents: 1967
Classes: ['comp.graphics', 'rec.sport.baseball']

Vocabulary Size: 5000

Class: comp.graphics
Total Words: 72295
Zero Probability Words (MLE): 717

Class: rec.sport.baseball
Total Words: 39927
Zero Probability Words (MLE): 1637

MLE Estimation Completed

Dirichlet Prior alpha = 0.5
Class: comp.graphics
Zero Probability Words: 0
Class: rec.sport.baseball
Zero Probability Words: 0

Dirichlet Prior alpha = 1
Class: comp.graphics
Zero Probability Words: 0
Class: rec.sport.baseball
Zero Probability Words: 0

Dirichlet Prior alpha = 5
Class: comp.graphics
Zero Probability Words: 0
Class: rec.sport.baseball
Zero Probability Words: 0

Dirichlet Prior alpha = 10
Class: comp.graphics
Zero Probability Words: 0
Class: rec.sport.baseball
Zero Probability Words: 0

MAP Estimation Completed

MLE Performance
Accuracy: 0.9573170731707317

MAP alpha = 0.5
Accuracy: 0.9512195121951219

MAP alpha = 1
Accuracy: 0.9471544715447154

MAP alpha = 5
Accuracy